## Desafio 2 — Consumo de Dados (SWAPI) | Globo (Engenharia de Dados)

Este notebook implementa um pipeline simples de **extração → transformação → carga (ETL)** a partir da API pública do Star Wars (SWAPI), armazenando os dados em **SQLite** e gerando **insights via SQL**.

**Stack utilizada**
- Python (requests, pandas)
- SQLite (armazenamento)
- SQL (análises e insights)
- Notebook (Jupyter/Colab)

### 1. Bibliotecas, variáveis e configurações

Nesta seção são importadas as bibliotecas necessárias e definidos os endpoints que serão consumidos.

In [83]:
# Bibliotecas
import sqlite3
import time
from pathlib import Path

import pandas as pd
import requests

In [84]:
# Variáveis
url = 'https://swapi.dev/api'
endpoint = ['films', 'people', 'planets', 'species', 'starships', 'vehicles']

### 2. Funções auxiliares (extração com paginação e extração de ID)

Extração Paginada (`fetch_all`): Esta função percorre todos os links next de um endpoint até consolidar todos os registros em uma única lista, evitando perda de dados.

Tratamento de IDs (`extract_id`): Esta função é responsável por extrair somente o identificador numérico final das URLs para garantir o relacionamento entre os registros.

In [ ]:
def fetch_all(base_url: str, endpoint: str, sleep_s: float = 0.1) -> list[dict]:
    """
    Consome dados da API de forma paginada.
    
    Args:
        base_url: URL base da API (ex: https://swapi.dev/api).
        endpoint: O recurso específico a ser consumido (ex: 'people', 'starships').
        sleep_s: Tempo de espera entre requisições para evitar rate limit.
        
    Returns:
        Uma lista contendo todos os registros (dicts) encontrados no endpoint.
    """
    # Monta a URL inicial combinando a base e o endpoint
    next_url = f"{base_url}/{endpoint}/"
    out = []

    # Itera enquanto houver uma URL de próxima página retornada pela API
    while next_url:
        # Realiza a requisição GET com um limite de tempo (timeout) para evitar travamentos
        r = requests.get(next_url, timeout=30)
        
        # Garante que a função pare caso ocorra um erro de conexão ou permissão (4xx ou 5xx)
        r.raise_for_status()
        
        data = r.json()
        
        # Adiciona os resultados da página atual à lista
        out.extend(data["results"])
        
        # Atualiza a URL para a próxima página ou None, caso chegue ao fim
        next_url = data["next"]
        
        # Pausa controlada para evitar sobrecarga da API
        time.sleep(sleep_s)

    return out

# Função auxiliar para extrair o ID da URL
def extract_id(url_string):
    """
    Extrai o identificador numérico final de uma URL da SWAPI.
    
    Transforma strings no formato 'https://swapi.dev/api/people/1/' em inteiros (1).
    Essencial para a criação de Chaves Primárias (PK) e Estrangeiras (FK) no banco de dados.
    """
    if not url_string: 
        return None
    
    # Extrai o número entre as últimas barras da URL
    return int(url_string.split('/')[-2])